# <span style="color:green"> THIS CODE IS TO CALCULATE THE CENTERLINE OF AN STL MODEL AND THE DIAMETER OF THE PORES ALONG THE VERTICES OF THE CENTERLINE USING RAY CASTS

# Load libraries

In [34]:
import skeletor as sk
import trimesh as tm
import os
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from collections import defaultdict
import openpyxl
import math

In [35]:
#! /slicer/bin/./python-real -m pip install skeletor
#! /slicer/bin/./python-real -m pip install openpyxl


# Papermill parameters to be changed

## Load the STL file

## <span style="color:red">  note: Open the swc file using neuTube software

In [36]:
stl_path = 'cylinder/9_repaired.stl'

In [37]:
# Load your own STL file
mesh = tm.load_mesh(stl_path)

# Fix the mesh
fixed = sk.pre.fix_mesh(mesh, remove_disconnected=5, inplace=False)

# Skeletonazition of the model

In [ ]:
# Compute the skeleton
skel = sk.skeletonize.by_wavefront(fixed, waves=1, step_size=2)


# Visualize the skeleton overlaid on the original mesh
skel.show(mesh=True)

# save the skeleton (centerline)
output_dir = os.path.dirname(stl_path)
skel.save_swc(output_dir + "/skeleton.swc")


# FIGURE 1: MODEL + CENTERLINE
# Create a new figure
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

# Plot the mesh
ax.plot_trisurf(*fixed.vertices.T, triangles=fixed.faces, color='b', alpha=0.5)

# Plot the skeleton edges
for edge in skel.edges:
    start = skel.vertices[edge[0]]
    end = skel.vertices[edge[1]]
    ax.plot([start[0], end[0]], [start[1], end[1]], [start[2], end[2]], color='r')

# Set labels
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')

# Set equal aspect ratio
ax.set_box_aspect([1, 1, 1])

# Set limits for all axes
max_range = max(max(fixed.vertices.max(axis=0)), max(skel.vertices.max(axis=0)))
min_range = min(min(fixed.vertices.min(axis=0)), min(skel.vertices.min(axis=0)))
ax.set_xlim([min_range, max_range])
ax.set_ylim([min_range, max_range])
ax.set_zlim([min_range, max_range])

# Save the figure as an image file
fig.savefig(os.path.dirname(stl_path) + '/skeleton.png', dpi=300)  # Change the filename and dpi as needed


Skeletonizing:   0%|          | 0/472 [00:00<?, ?it/s]

# Compute the radii of the pores

In [30]:
# compute radii of pores as casted rays at each vertex of the centerline in the direction perpendicular to the direction of the centerline
sk.post.radii(skel, fixed, method='ray', projection='tangents') # tangets: dont take distance along the path (no length of pore only diameter)

skel.swc.to_excel(output_dir + "/data.xlsx", sheet_name = "radii", index=False)

# Calculate minimum, maximum, and mean radius
radii_data = skel.swc['radius']

min_radius = radii_data.min()
max_radius = radii_data.max()
mean_radius = radii_data.mean()

print(f"Minimum diameter: {min_radius*2}")
print(f"Maximum diameter: {max_radius*2}")
print(f"Mean diameter: {mean_radius*2}")

Minimum diameter: 0.6832563245032252
Maximum diameter: 1.1806753839188961
Mean diameter: 0.90606568183058


# Compute the path lengths of the pores

## Find the all the possible paths

In [31]:
# Create a dictionary to count the occurrences of each vertex
vertex_count = defaultdict(int)

# Create an adjacency list to represent the graph
graph = defaultdict(list)

skel_vertices = skel.vertices
skel_edges = skel.edges


# Calculate distances and populate the adjacency list and vertex count
for edge in skel_edges:
    v1, v2 = edge
    start = np.array(skel_vertices[v1])
    end = np.array(skel_vertices[v2])
    distance = np.linalg.norm(end - start)
    graph[v1].append((v2, distance))
    graph[v2].append((v1, distance))
    vertex_count[v1] += 1
    vertex_count[v2] += 1

# Identify the leaf vertices (vertices of degree 1 meaning they are that are connected to one and only one edge)
leaf_vertices = [vertex for vertex, count in vertex_count.items() if count == 1]

# Function to perform DFS (depth-first search) and find all paths from start to any leaf vertex
def dfs(current, target, visited, path, path_distance, all_paths):
    visited.add(current)
    path.append(current)

    if current == target:
        all_paths.append((path.copy(), path_distance))
    else:
        for neighbor, distance in graph[current]:
            if neighbor not in visited:
                dfs(neighbor, target, visited, path, path_distance + distance, all_paths)

    path.pop()
    visited.remove(current)

# Find all unique paths between leaf vertices
all_paths = []
for start_vertex in leaf_vertices:
    for end_vertex in leaf_vertices:
        if start_vertex != end_vertex:
            dfs(start_vertex, end_vertex, set(), [], 0, all_paths)


## Remove duplicate paths (since paths can be reversed)

In [32]:
unique_paths = []
seen = set()
for path, path_distance in all_paths:
    path_tuple = tuple(path)
    if path_tuple not in seen and tuple(reversed(path)) not in seen:
        unique_paths.append((path, path_distance))
        seen.add(path_tuple)

# Calculate the distance for each unique path and format the output
paths_with_distances = []
for path, _ in unique_paths:
    total_distance = 0
    for i in range(len(path) - 1):
        for neighbor, distance in graph[path[i]]:
            if neighbor == path[i + 1]:
                total_distance += distance
                break
    paths_with_distances.append((path, total_distance))



# Save the results

In [33]:
# Define the maximum number of rows per sheet
MAX_ROWS = 1048576

# Open the Excel file
workbook = openpyxl.load_workbook(output_dir + "/data.xlsx")

# Calculate the number of sheets required
num_sheets = math.ceil(len(paths_with_distances) / (MAX_ROWS - 1))

# Write data to multiple sheets
for sheet_index in range(num_sheets):
    # Create a new worksheet or get the existing "paths" worksheet
    sheet_name = f'paths_{sheet_index + 1}'
    if sheet_name not in workbook.sheetnames:
        worksheet = workbook.create_sheet(title=sheet_name)
    else:
        worksheet = workbook[sheet_name]

    # Write headers
    worksheet['A1'] = 'Index'
    worksheet['B1'] = 'Path'
    worksheet['C1'] = 'Distance'

    # Write data to the worksheet
    start_index = sheet_index * (MAX_ROWS - 1)
    end_index = start_index + (MAX_ROWS - 1)
    for index, (path, distance) in enumerate(paths_with_distances[start_index:end_index], start=2):
        path_str = ' -> '.join(str(vertex) for vertex in path)
        worksheet[f'A{index}'] = start_index + index - 1
        worksheet[f'B{index}'] = path_str
        worksheet[f'C{index}'] = distance

# Save the workbook
workbook.save(output_dir + "/data.xlsx")